## SECTION 1: Desensitisation 


In this section, we take the stored information from our JSON file, and stip it of any sensative names
The AUS Goverment contact said that these needed to be stript before publishing any information

In [1]:
# pip install torch


# 1.1 Load JSON data 

In [4]:
import json
import pandas as pd 

# Load JSON data from a file
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)  # Parse the JSON file
    return data

# Example usage
file_path = '../data/raw/raw_scraped_dataset.json'  # Replace with the path to your JSON file
json_data = load_json(file_path)

# Print loaded JSON data
# print(json_data)
print(json_data[:5])  # This will print only the first 5 items if it's a list

# Now load into a dataframe
# Flatten the nested structure and create a DataFrame
flattened_data = [item[0] for item in json_data]  # Flatten out the inner list
df_all = pd.DataFrame(flattened_data)


# Add a unique column to each URL_MAIN (you can use a hash function or unique string)
df_all['URL_ID'] = pd.factorize(df_all['URL_MAIN'])[0] + 1


# Add an overall row ID (this will be the index of the DataFrame, starting from 1)
df_all['Row_ID'] = df_all.index + 1  # Adding a simple row ID, starting from 1

df_all = df_all[['Row_ID', 'URL_ID'] + [col for col in df_all.columns if col not in ['URL_ID', 'Row_ID']]]


df_all.to_csv("../data/processed/processed_scraped_dataset.csv", escapechar='\\', quoting=1)

[[{'Owner': 'CPA Australia', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/324', 'Raw_Scraped_Text': ' \n \nAI Consultation Team  \nIndustry House \nDepartment of Industry, Science and Resources \n10 Binara St \nCanberra ACT  2600 \n \n9 October 2024 \n  \nDear Helen, \nProposals paper for introducing mandatory guardrails for AI in high-risk settings \nCPA Australia represents the diverse interests of more than 173,000 accounting and finance professionals, working in \nover 100 countries and regions around the world. We make this submission in response to the  Proposals Paper for \nintroducing mandatory guardrails for AI in high-risk settings  on behalf of our members and in the broader public \ninterest. \nCPA Australia supports the development of regulatory options for the safe and responsible use of AI in high-risk \nsituations. Such regulatory options need to strike a balance between protecting the community from damaging uses of \nAI and enco

In [6]:
fax = pd.read_csv("../data/processed/processed_scraped_dataset.csv")

# 1.2 Desensitisation 


In [12]:
# For each piece of text, run though LLM



from openai import OpenAI
def LLM_CALL(text, owner):
    
    Prompt = f'''
        ## CONTENT
        {text}

        ## Instructions
        The above content is a submission by {owner} in response to the Australian Government's introduction of mandatory guardrails for AI in high-risk settings. Your task is to anonymize any sensitive information in the provided text according to the following steps:

        1. Read the entire content carefully.
        2. Identify and replace the following types of sensitive information with '###':
            - Personal names (e.g., individuals' full names).
            - Company names or trademarks.
            - Specific addresses (e.g., street addresses, PO Boxes).
            - Contact details (e.g., phone numbers, fax numbers, email addresses).
            - Any other personally identifiable information (PII) or confidential data.
        3. Ensure that all instances of sensitive information are replaced consistently throughout the entire text.
        4. Do not modify the rest of the content, preserving its meaning while removing any details that could identify individuals, businesses, or sensitive data.    
        '''
    
    client = OpenAI(
      base_url = "https://integrate.api.nvidia.com/v1",
      api_key = "$API_KEY_REQUIRED_IF_EXECUTING_OUTSIDE_NGC"
    )

    completion = client.chat.completions.create(
      model="meta/llama-3.1-70b-instruct",
      messages=[{"role":"user","content":Prompt}],
      temperature=0.2,
      top_p=0.7,
      max_tokens=9000000,
      stream=True
    )

    for chunk in completion:
        if chunk.choices[0].delta.content is not None:
            print(chunk.choices[0].delta.content, end="")


## NEED TO ADD DESENSITISED below/above

In [13]:
eg = df_all["Raw_Scraped_Text"][0]
LLM_CALL(eg, 'CFA')

APIConnectionError: Connection error.

# JUNK

In [5]:
from nemo_curator.datasets import DocumentDataset
from nemo_curator.utils.distributed_utils import read_data, write_to_disk, get_client
from nemo_curator.utils.file_utils import get_batched_files
from nemo_curator.modules.modify import Modify
from nemo_curator.modifiers.pii_modifier import PiiModifier
import pandas as pd

import dask.dataframe as dd
import pandas as pd


modifier = PiiModifier(
    language="en",
    supported_entities=["PERSON", "EMAIL_ADDRESS"],
    anonymize_action="replace",
    batch_size=1000,
    device="gpu")

# Sample data with 'text' column
sample_data = [
    {"text": "Alice is a person and her email is alice@example.com"}]
# sample_data = json_data[0][0]
df = pd.DataFrame(sample_data, index=[0])
# text_df = df["Raw_Scraped_Text"]
# text_df = text_df.rename(columns={"Raw_Scraped_Text": "text"}, inplace=True)

# Convert pandas DataFrame to Dask DataFrame
ddf = dd.from_pandas(df, npartitions=2)


dataset = DocumentDataset(ddf)
modify = Modify(modifier)
modified_dataset = modify(dataset)
print(modified_dataset)


# Extract the modified DataFrame (the underlying data)
modified_df = modified_dataset.df

# Now access the 'text' column and print the modified text
modified_texts = modified_df['text']
print(modified_texts)

Dask Series Structure:
npartitions=1
0    object
0       ...
Name: text, dtype: object
Dask Name: getitem, 5 graph layers


In [9]:
import dask.dataframe as dd
import pandas as pd
from nemo_curator.datasets import DocumentDataset
from nemo_curator.utils.distributed_utils import read_data, write_to_disk, get_client
from nemo_curator.utils.file_utils import get_batched_files
from nemo_curator.modules.modify import Modify
from nemo_curator.modifiers.pii_modifier import PiiModifier


# Define the PiiModifier
modifier = PiiModifier(
    language="en",
    supported_entities=["PERSON", "EMAIL_ADDRESS"],
    anonymize_action="replace",
    batch_size=1000,
    device="gpu"
)

# Sample data with 'text' column
sample_data = [
    {"text": "Alice is a person and her email is alice@example.com"}
]

# Convert sample data to pandas DataFrame
df = pd.DataFrame(sample_data, index=[0])

# Convert pandas DataFrame to Dask DataFrame
ddf = dd.from_pandas(df, npartitions=2)

# Create DocumentDataset
dataset = DocumentDataset(ddf)

# Apply the PiiModifier
modify = Modify(modifier)
modified_dataset = modify(dataset)

write_to_disk(modified_dataset.df,
                "../data/processed/new_text.json",
                output_type="jsonl"
                )




/home/g05_u05/.local/lib/python3.10/site-packages/cudf/utils/_ptxcompiler.py:64: UserWarning: Error getting driver and runtime versions:

stdout:



stderr:

Traceback (most recent call last):
  File "<string>", line 7, in <module>
  File "/home/g05_u05/.local/lib/python3.10/site-packages/numba/cuda/cudadrv/runtime.py", line 111, in get_version
    self.cudaRuntimeGetVersion(ctypes.byref(rtver))
  File "/home/g05_u05/.local/lib/python3.10/site-packages/numba/cuda/cudadrv/runtime.py", line 65, in __getattr__
    self._initialize()
  File "/home/g05_u05/.local/lib/python3.10/site-packages/numba/cuda/cudadrv/runtime.py", line 51, in _initialize
    self.lib = open_cudalib('cudart')
  File "/home/g05_u05/.local/lib/python3.10/site-packages/numba/cuda/cudadrv/libs.py", line 65, in open_cudalib
    return ctypes.CDLL(path)
  File "/usr/lib/python3.10/ctypes/__init__.py", line 374, in __init__
    self._handle = _dlopen(self._name, mode)
OSError: libcudart.so: cannot open shared object file: 

ValueError: Cannot use GPU, CuPy is not installed

In [17]:
pip install cupy-cuda12x


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
